In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading the Dataset

In [26]:
import numpy as np

def load_raw_signals(folder_path, filenames): # a function to load the dataset
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt" # makes the file path fro every file
        data = np.loadtxt(file_path) #Loads the data from the file into an Numpy array
        signals.append(data) # appends the list we created with the new readings
    
    return np.transpose(np.array(signals), (1, 2, 0)) # for our project it requires we have the array in (sample, step, feature) format

# file names we want to load
filenames = [
    'body_acc_x_train', 'body_acc_y_train', 'body_acc_z_train',
    'body_gyro_x_train', 'body_gyro_y_train', 'body_gyro_z_train'
]

#The absolut path for the folder in our dataset where the files are located
raw_data_path = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
X_raw = load_raw_signals(raw_data_path, filenames)

print(f"Raw 6-axis data shape: {X_raw.shape}") #printing the shape

Raw 6-axis data shape: (7352, 128, 6)


In [ ]:
# Normalizing the data
import numpy as np

# --- STEP 1: Load everything first ---
def load_raw_signals(folder_path, filenames):
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt"
        data = np.loadtxt(file_path)
        signals.append(data)
    # Returns (samples, 128, 6)
    return np.transpose(np.array(signals), (1, 2, 0))

filenames_train = [f.replace('_test', '_train') for f in filenames]
raw_data_path_train = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals/'

x_train_raw = load_raw_signals(raw_data_path_train, filenames_train)
X_test_raw = load_raw_signals(raw_data_path_test, filenames_test)

# --- STEP 2: Calculate Normalization ONLY on Training Data ---
# This prevents "Data Leakage"
mean = np.mean(x_train_raw, axis=(0, 1))
std = np.std(x_train_raw, axis=(0, 1))

# --- STEP 3: Normalize BOTH sets using Training constants ---
x_train = (x_train_raw - mean) / (std + 1e-8)
X_test = (X_test_raw - mean) / (std + 1e-8) # This was the missing step!

# --- STEP 4: Load Labels ---
y_train = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/y_train.txt', dtype=int) - 1
y_test = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt', dtype=int) - 1

print(f"Normalized x_train shape: {x_train.shape}")
print(f"Normalized X_test shape: {X_test.shape}")

# Coding the ID-CNN Architecture

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models
model = keras.Sequential(name="HAR_1DCNN") 
model.add(layers.Input(shape=(128, 6))) # loading an empty stack to which we'll add layers
model.add(layers.Conv1D(   #Now we're adding layers to this 
    filters=64,    #this arg defines how many local features we'll look for
    kernel_size=5, #this is essentially our window size which we slide over our 128 timesteps to extract features
    activation='relu', #rectified linear unit introduces non-linearity basically acting as a gate to decide which information is important enough to be passed to the next layer
    padding= 'same',   #This makes sure that our output is the same length as our input adds zeros so the kernel can center itself
    input_shape=(128,6) #This defines the size of our input passed 
))
model.add(layers.MaxPooling1D(pool_size=4))
model.add(layers.Dropout(rate=0.5)) # Increased slightly for better regularization
model.add(layers.Flatten())

# 1. Add a "Brain" (Hidden Layer)
model.add(layers.Dense(64, activation='relu')) 

# 2. Add another Dropout here to stop the Dense layer from over-memorizing
model.add(layers.Dropout(0.5))

# 3. Final Output Layer (MUST be Softmax for 6 classes)
model.add(layers.Dense(6, activation='softmax'))
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test),
    shuffle=True,
)